In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import re

# =========================================
# 1. CONFIG
# =========================================

catalog_name = "workspace"
schema_name = "default"

# Source volume path from your screenshot
base_path = f"/Volumes/{catalog_name}/{schema_name}/sephora_surfers"

# Bronze schema to store Delta tables
bronze_schema = f"{catalog_name}.{schema_name}"

# Optional: create schema if needed
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

# =========================================
# 2. HELPER FUNCTIONS
# =========================================

def clean_column_name(col_name: str) -> str:
    """
    Standardize column names for Bronze:
    - lowercase
    - replace non-alphanumeric with underscore
    - collapse multiple underscores
    - strip leading/trailing underscores
    """
    col_name = col_name.strip().lower()
    col_name = re.sub(r'[^a-z0-9]+', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    col_name = col_name.strip('_')
    return col_name

def standardize_columns(df):
    return df.toDF(*[clean_column_name(c) for c in df.columns])

def add_bronze_metadata(df, source_file_label: str):
    return (
        df.withColumn("bronze_ingestion_ts", F.current_timestamp())
          .withColumn("bronze_source_file", F.col("_metadata.file_path"))
          .withColumn("bronze_source_label", F.lit(source_file_label))
          .withColumn("bronze_load_date", F.current_date())
    )

# =========================================
# 3. READ PRODUCT_INFO.CSV
# =========================================

product_info_path = f"{base_path}/product_info*.csv"

df_product_info = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("multiLine", "true")
         .option("escape", '"')
         .option("quote", '"')
         .option("inferSchema", "false")   # keep raw as strings in Bronze
         .load(product_info_path)
)

df_product_info = standardize_columns(df_product_info)
df_product_info = add_bronze_metadata(df_product_info, "product_info")

display(df_product_info.limit(5))

df_product_info.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_sephora_product_info")

In [0]:
# =========================================
# 4. READ COSMETIC_P.CSV
# =========================================

cosmetic_path = f"{base_path}/cosmetic_p*.csv"

df_cosmetic = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("multiLine", "true")
         .option("escape", '"')
         .option("quote", '"')
         .option("inferSchema", "false")
         .load(cosmetic_path)
)

df_cosmetic = standardize_columns(df_cosmetic)
df_cosmetic = add_bronze_metadata(df_cosmetic, "cosmetic_p")

display(df_cosmetic.limit(5))

df_cosmetic.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_skincare_cosmetic_p")

In [0]:
# =========================================
# 5. READ ALL REVIEW FILES
# =========================================

review_paths = f"{base_path}/reviews_*.csv"

df_reviews = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("multiLine", "true")
         .option("escape", '"')
         .option("quote", '"')
         .option("inferSchema", "false")
         .load(review_paths)
)

df_reviews = standardize_columns(df_reviews)
df_reviews = add_bronze_metadata(df_reviews, "reviews_combined")

display(df_reviews.limit(5))

df_reviews.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_sephora_reviews")